In [ ]:
import pandas as pd
import os
import numpy as np

In [ ]:
# FOR GOOGLE COLAB
#from google.colab import drive
#drive.mount('/content/gdrive')
#data_path = '/content/gdrive/MyDrive/SIADS_696_Milestone_II/Datasets/EIA_files'

# FOR JUPYTERLAB
data_path = os.getcwd()

In [ ]:
# Read SEDS production data (rename state column to match consumption data)
file_prod = os.path.join(data_path, 'Prod_dataset.xlsx')
df_prod = pd.read_excel(file_prod, sheet_name='Data').rename(columns={"StateCode": "State"})

# Read SEDS consumption data
file_cons = os.path.join(data_path, 'use_all_btu.xlsx')
df_cons = pd.read_excel(file_cons, sheet_name='Data')

# Read hand picked categories of data
file_cat = os.path.join(data_path, 'SEDS_categories.xlsx')
df_cat = pd.read_excel(file_cat)

# Printing
display(df_prod)
display(df_cons)
display(df_cat.head())

In [ ]:
# Combine prod/cons datasets
df = pd.concat([df_prod, df_cons], ignore_index=True)
df

In [ ]:
# Filter for MSN codes, drop Data_Status, and remove non-states
df_filtered = df[df['MSN'].isin(df_cat['MSN Code'])].drop(columns=['Data_Status'])
df_filtered = df_filtered[~df_filtered['State'].isin(['US', 'X3', 'X5'])]
df_filtered

In [ ]:
# Check some stats for dirty data (all are clean)
df_filtered.info()

In [ ]:
# Transpose Years/MSN, but keep State and filter for year range (optional)
id_vars = ['State', 'MSN']
value_vars = [col for col in df_filtered.columns if col not in id_vars]
df_filtered_melted = df_filtered.melt(id_vars=id_vars, value_vars=value_vars, var_name='Year', value_name='Value')
df_filtered_transposed = df_filtered_melted.pivot_table(index=['State', 'Year'], columns='MSN', values='Value')
df_filtered_transposed = df_filtered_transposed.reset_index()
#df_filtered_transposed = df_filtered_transposed[df_filtered_transposed['Year'].isin(range(1980, 2025))]
df_filtered_transposed

In [ ]:
# Show some stats
display(df_filtered_transposed.sample(10))
display(df_filtered_transposed.describe())

In [ ]:
# Save df to CSV
output_file_path = os.path.join(data_path, 'df_seds_cleaned.csv')
df_filtered_transposed.to_csv(output_file_path, index=False)